In [2]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("Qwen/Qwen3-0.6B")

print("Model loaded!")
print(f"Layers: {model.cfg.n_layers}")
print(f"Hidden dimension: {model.cfg.d_model}")

/tmp/ipykernel_12318/1457788674.py:3: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("Qwen/Qwen3-0.6B")
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen3-0.6B into HookedTransformer
Model loaded!
Layers: 28
Hidden dimension: 1024


In [10]:
prompt = """Solve this problem step by step.

Alice is older than Bob.
Bob is older than Charlie.

Question: Who is the oldest?

Explain your reasoning and then give the final answer."""

tokens = model.to_tokens(prompt)

logits, cache = model.run_with_cache(tokens)

print("Number of tokens:", tokens.shape[1])
print("Cached activation keys:", len(cache))

Number of tokens: 38
Cached activation keys: 677


In [11]:
for layer in [0, 7, 14, 21, 27]:
    activation = cache[f"blocks.{layer}.hook_resid_post"]
    print(
        f"Layer {layer}:",
        activation.shape
    )

Layer 0: torch.Size([1, 38, 1024])
Layer 7: torch.Size([1, 38, 1024])
Layer 14: torch.Size([1, 38, 1024])
Layer 21: torch.Size([1, 38, 1024])
Layer 27: torch.Size([1, 38, 1024])


In [4]:
clean_prompt = """Alice is older than Bob.
Bob is older than Charlie.
Who is the oldest?"""

corrupt_prompt = """Alice is younger than Bob.
Bob is older than Charlie.
Who is the oldest?"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

print("Clean:")
print(model.to_str_tokens(clean_tokens))

print("\nCorrupted:")
print(model.to_str_tokens(corrupt_tokens))

Clean:
['Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n', 'Who', ' is', ' the', ' oldest', '?']

Corrupted:
['Alice', ' is', ' younger', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n', 'Who', ' is', ' the', ' oldest', '?']


In [5]:
clean_logits = model(clean_tokens)
corrupt_logits = model(corrupt_tokens)

clean_next = clean_logits[0, -1].argmax()
corrupt_next = corrupt_logits[0, -1].argmax()

print("Clean prediction:", repr(model.to_string(clean_next)))
print("Corrupted prediction:", repr(model.to_string(corrupt_next)))

Clean prediction: ' A'
Corrupted prediction: ' A'


In [6]:
clean_prompt = """Answer the question with exactly one name.

Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer: Alice"""

corrupt_prompt = """Answer the question with exactly one name.

Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer: Bob"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

print("Clean:")
print(model.to_str_tokens(clean_tokens))

print("\nCorrupted:")
print(model.to_str_tokens(corrupt_tokens))

Clean:
['Answer', ' the', ' question', ' with', ' exactly', ' one', ' name', '.\n\n', 'Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Answer', ':', ' Alice']

Corrupted:
['Answer', ' the', ' question', ' with', ' exactly', ' one', ' name', '.\n\n', 'Alice', ' is', ' younger', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Answer', ':', ' Bob']


In [10]:
clean_prompt = """Answer with exactly one name.

Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer:"""

corrupt_prompt = """Answer with exactly one name.

Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer:"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

clean_logits = model(clean_tokens)
corrupt_logits = model(corrupt_tokens)

# Probability of the two candidate answers
for name in [" Alice", " Bob"]:
    token = model.to_single_token(name)
    clean_prob = clean_logits[0, -1].softmax(-1)[token].item()
    corrupt_prob = corrupt_logits[0, -1].softmax(-1)[token].item()

    print(f"{name}: clean={clean_prob:.4f}, corrupted={corrupt_prob:.4f}")

 Alice: clean=0.1046, corrupted=0.0604
 Bob: clean=0.0052, corrupted=0.0048
